# PDD Mobility Scanner — Format `trail_data.csv` to dashboard XLSX

Slices `trail_data.csv` into 5-second waypoints and produces the dashboard-compatible xlsx with `trail_data` + `waypoint_photos` sheets.

No video, no YOLO, no `video_sync.csv` upload — waypoint timing is taken straight from the `ms` column (relative to the first row).

Surface / obstacle / infrastructure columns are filled with placeholder values; populate them later from your separate inference pipeline if needed.

## Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
from google.colab import files

## Step 2: Upload `trail_data.csv`

In [ ]:
uploaded = files.upload()
trail_csv = next(f for f in uploaded.keys() if f.lower().endswith('.csv'))
print(f'Trail CSV: {trail_csv}')

## Step 3: Load sensor data

`video_sec` is derived from `ms` relative to the first sensor row, since there's no sync file.

In [ ]:
df = pd.read_csv(trail_csv)
df['video_sec'] = (df['ms'] - df['ms'].iloc[0]) / 1000.0
print(f'Loaded {len(df)} sensor rows, duration {df["video_sec"].max():.1f}s')

## Step 4: Assign waypoints (5-second windows)

In [ ]:
WAYPOINT_SEC = 5
df['wp'] = np.maximum(0, np.floor(df['video_sec'] / WAYPOINT_SEC).astype(int))
n_waypoints = df['wp'].nunique()
print(f'{n_waypoints} waypoints over {len(df)} rows '
      f'(avg {len(df) / n_waypoints:.0f} rows per waypoint)')

## Step 5: Build `trail_data` and `waypoint_photos` DataFrames

`image_file` is set to `img_NNNN.jpg` placeholders so the dashboard schema is preserved. Surface/obstacle/infrastructure columns get fallback values.

In [ ]:
df['image_file']   = df['wp'].map(lambda wp: f'img_{int(wp):04d}.jpg')
df['surface_type'] = 'No surface detected'
df['obstacle']     = 'None detected'
df['substructure'] = 'None detected'

TARGET_COLUMNS = [
    'ms', 'utc', 'wp',
    'ax', 'ay', 'az',
    'gvx', 'gvy', 'gvz',
    'gx', 'gy', 'gz',
    'qw', 'qx', 'qy', 'qz',
    'mx', 'my', 'mz',
    'tof_mm', 'tof_status',
    'lat', 'lng', 'alt', 'speed', 'hdg', 'sats', 'hdop',
    'cal_sys', 'cal_gyro', 'cal_accel', 'cal_mag',
    'image_file', 'surface_type', 'obstacle', 'substructure',
]

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
assert not missing, f'Missing columns from input: {missing}'

out_df = df[TARGET_COLUMNS]

waypoint_photos_df = (
    out_df[['wp', 'image_file']]
    .drop_duplicates(subset='wp')
    .sort_values('wp')
    .reset_index(drop=True)
)
waypoint_photos_df['surface_type']   = 'No surface detected'
waypoint_photos_df['obstacles']      = 'None detected'
waypoint_photos_df['infrastructure'] = 'None detected'

print(f'trail_data:      {len(out_df)} rows, {len(out_df.columns)} columns')
print(f'waypoint_photos: {len(waypoint_photos_df)} rows')
waypoint_photos_df.head(10)

## Step 6: Write XLSX and download

In [ ]:
out_dir = 'output'
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(out_dir)

xlsx_path = os.path.join(out_dir, 'trail_data.xlsx')
with pd.ExcelWriter(xlsx_path) as writer:
    out_df.to_excel(writer, sheet_name='trail_data', index=False)
    waypoint_photos_df.to_excel(writer, sheet_name='waypoint_photos', index=False)
print(f'Wrote {xlsx_path} (sheets: trail_data, waypoint_photos)')

files.download(xlsx_path)